In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source":"mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are imdependent and often  enjow their own space.",
        metadata={"source":"mammal-pets-doc"},
    ),
    Document(
        page_content="GoldFish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source":"fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source":"bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source":"mammal-pets-doc"},
    ),
]

In [2]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are imdependent and often  enjow their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='GoldFish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [18]:
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

llm = ChatGroq(groq_api_key=groq_api_key,model="llama-3.1-8b-instant")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000205F3FA13F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000205F3997790>, model_name='llama-3.1-8b-instant')

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1623.14it/s]


In [20]:
from langchain_chroma import Chroma

vectorstore=Chroma.from_documents(documents,embedding=embeddings)
vectorstore

In [21]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='84daa7a2-5cea-4e60-be84-df65a27f1972', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are imdependent and often  enjow their own space.'),
  0.8950580358505249),
 (Document(id='02de58d1-f1ca-494d-801a-62b396bf76bf', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are imdependent and often  enjow their own space.'),
  0.8950580358505249),
 (Document(id='2e33dada-655d-4a4a-a385-bfd1264c7c7d', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740898847579956),
 (Document(id='55ec15e5-080c-4ac4-845f-fd2dc2c1792d', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740898847579956)]

In [22]:
await vectorstore.asimilarity_search("cat")

[Document(id='84daa7a2-5cea-4e60-be84-df65a27f1972', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are imdependent and often  enjow their own space.'),
 Document(id='02de58d1-f1ca-494d-801a-62b396bf76bf', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are imdependent and often  enjow their own space.'),
 Document(id='2e33dada-655d-4a4a-a385-bfd1264c7c7d', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='55ec15e5-080c-4ac4-845f-fd2dc2c1792d', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [23]:
### Retrivers
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriver=RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriver.batch(["cat","dog","parrot"])

[[Document(id='02de58d1-f1ca-494d-801a-62b396bf76bf', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are imdependent and often  enjow their own space.')],
 [Document(id='55ec15e5-080c-4ac4-845f-fd2dc2c1792d', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')],
 [Document(id='507bd8ae-bf0a-48f9-bb7c-f8cb465de8a3', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]]

In [ ]:
## prefer this approach
retriver=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriver.batch(["cat","dog","parrot"])


[[Document(id='02de58d1-f1ca-494d-801a-62b396bf76bf', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are imdependent and often  enjow their own space.')],
 [Document(id='55ec15e5-080c-4ac4-845f-fd2dc2c1792d', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')],
 [Document(id='507bd8ae-bf0a-48f9-bb7c-f8cb465de8a3', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]]

In [ ]:
### integrating retriver and chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using provided context only.
{question}

Context:
{context}
"""
prompt=ChatPromptTemplate.from_messages([("human",message)])

rag_chain={"context":retriver, "question":RunnablePassthrough()}|prompt|llm

response=rag_chain.invoke("Tell me about cats")
print(response.content)

Based on the provided context, here's what we can say about cats:

Cats are independent and often enjoy their own space.
